# 用 Word2Vec 做歌曲推薦：把「歌單」當成「句子」，把「歌曲」當成「單字」


## 這份程式要做什麼

給一首歌，找出「最像它」的 5 首歌，做成簡單的**歌曲推薦**。

```
電台歌單資料（每一行是一份歌單，裡面是歌曲編號）
        │
        ▼  Word2Vec：學習每首歌的「向量」
        │   （常出現在同一份歌單裡的歌，向量會很接近）
        ▼
每首歌一個 32 維向量
        │
        ▼  找向量最接近的歌（餘弦相似度）
        ▼
推薦：和這首歌最像的 5 首歌
```

## 核心想法：歌單就是句子，歌曲就是單字

Word2Vec 原本是用來學習**詞向量**的：

> 常出現在相近上下文裡的詞，意思也相近。

把這個想法搬到音樂上：

| 文字世界 | 音樂世界（這份程式） |
|---|---|
| 一個詞（word） | **一首歌** |
| 一個句子（sentence） | **一份歌單（playlist）** |
| 上下文（周圍的詞） | **同一份歌單裡的其他歌** |
| 意思相近的詞 | **風格相近、常被一起播放的歌** |

所以我們**完全不需要知道歌的內容**（曲風、歌手、歌詞），只靠「誰和誰常常一起出現」，就能學出歌曲之間的相似度。這種做法有時叫 **item2vec**。

## 這屬於哪一類技術

| | Word2Vec（這份） | 你之前學的 Sentence-Transformers |
|---|---|---|
| 向量的對象 | 單一個「項目」（詞 / 歌） | 一整句話 |
| 是否看上下文 | **不看**：每個項目只有**一個固定向量** | 會看：同一個詞在不同句子裡向量不同 |
| 模型大小 | 很小，幾秒到幾分鐘就能訓練 | 大型 Transformer |
| 訓練資料 | 只需要「共現」（誰和誰一起出現） | 需要大量文字或配對 |
| 適合 | 推薦系統、快速的相似度 | 語意搜尋、文字理解 |

兩者背後的概念相同：**用向量表示事物，距離近就代表相似**。Word2Vec 是更早、更簡單的版本，也是理解 embedding 的最好起點。

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `urllib.request` | 從網路下載資料（`urlopen`） |
| `pandas` | 用表格（DataFrame）整理歌曲資訊 |
| `gensim.models.Word2Vec` | **訓練 Word2Vec 的工具** |
| `numpy` | 處理陣列（後面用來取出推薦歌曲的編號） |

第一行 `#使用.venv313` 只是你寫給自己的註解，不影響執行。

In [1]:
#使用.venv313

from urllib import request
import pandas as pd
from gensim.models import Word2Vec
import numpy as np

## 步驟 1：定義推薦函式

這個函式放在最前面，但它要用到後面才建立的 `model` 和 `songs_df`。這沒問題：函式**定義**時不會去找這兩個變數，等**被呼叫**時才去找，那時它們已經存在了。

```python
def print_recommendation(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=5)
    )[:, 0]

    return songs_df.iloc[similar_songs]
```

### 逐步拆解

**① `model.wv.most_similar(positive=str(song_id), topn=5)`**

| 部分 | 說明 |
|---|---|
| `model.wv` | 訓練好的**詞向量**（wv = word vectors），存著每首歌的向量 |
| `most_similar(...)` | 找出向量最接近的項目（用**餘弦相似度**） |
| `positive=str(song_id)` | 以這首歌為基準。**要轉成字串**，因為歌單裡的歌曲編號是字串（見步驟 2） |
| `topn=5` | 只要最像的 5 個 |

它回傳一個 list，每個元素是 `(歌曲編號, 相似度)` 的組合，編號是**字串**。

**② `np.array(...)[:, 0]`：只取編號那一欄**

我用小資料實測（沙箱，numpy 2.4.4）：

```
np.array([('3167', 0.91), ('2147', 0.88), ('1930', 0.85)])
  → dtype <U32，形狀 (3, 2)        ← 整個陣列被轉成字串型別，連相似度也變成字串
  → [:, 0] = ['3167' '2147' '1930']  ← 取出所有列的第 0 欄，也就是編號
```

`[:, 0]` 的意思是「**所有列、第 0 欄**」：冒號代表全部的列，`0` 代表第一欄。這裡只想要編號，不要相似度，所以取第 0 欄。

**③ `songs_df.iloc[similar_songs]`：用編號查歌曲資訊**

用編號去 `songs_df` 查出歌名與歌手。這一步有個值得注意的細節，說明如下。

### ⚠️ `iloc` 與 `loc` 的差別（我有實測）

| | `.loc[...]` | `.iloc[...]` |
|---|---|---|
| 依據 | **標籤**（index 的值） | **位置**（第幾列） |
| 本程式的 index | 歌曲編號（字串，例如 `'2172'`） | — |

這份程式用的是 `.iloc`，但傳進去的是**編號的字串陣列**。我在沙箱實測（pandas 2.2.3 與 3.0.2 都是）：`.iloc` 對這種「看起來像數字的字串」**會當成整數位置**使用，並不會報錯。

為什麼你的程式能正確運作？因為這個資料集的歌曲編號是 **0, 1, 2, ... 連續編號、依檔案順序排列**，所以**「第幾列」剛好等於「編號」**，位置和標籤碰巧一致。

這是一個**巧合**：

- `.iloc` 本來是設計給整數位置用的，用字串當位置不是它的正式用法，**不要依賴這個行為**
- 如果編號不是連續的（或表格被重新排序、刪掉某些列），`.iloc` 就會**悄悄取到錯的歌**，而且不報錯
- 用標籤查歌曲，正確的是 `.loc`：`songs_df.loc[similar_songs]`

我在沙箱用「標籤和位置刻意不一致」的小表格實測：`.iloc` 回傳的是**位置**對應的列，`.loc` 回傳的才是**標籤**對應的列。你的版本若不同，請以實際為準。

### 你註解掉的 `self-write` 版本

```python
similar_song = []
for song, _ in model.wv.most_similar(positive=str(song_id), topn=5):
    similar_song.append(song)
return(songs_df.iloc[similar_song])
```

這是用迴圈做同樣的事：逐一取出 `(編號, 相似度)` 組合，只留編號（`_` 是「不需要這個值」的慣用寫法），最後同樣用 `.iloc` 查。兩種寫法結果相同，迴圈版比較好讀，`np.array(...)[:, 0]` 版比較精簡。

### 兩個小提醒

- 函式叫 `print_recommendation`，但它**沒有 print，而是回傳 DataFrame**（是呼叫的人才 `print`）。名稱和行為不太一致，叫 `get_recommendation` 會更貼切。
- 它依賴全域變數 `model` 和 `songs_df`，沒有放在參數裡。短程式沒問題，但程式變大時，「隱藏的依賴」會讓人不容易看出函式需要什麼。

In [ ]:
#========================================================================
def print_recommendation(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=5)
    )[:, 0]

    return songs_df.iloc[similar_songs]

# ===========self-write==================
# def print_recommendation(song_id):
#     similar_song = []
#     for song, _ in model.wv.most_similar(positive=str(song_id), topn=5): 
#         similar_song.append(song)

#     return(songs_df.iloc[similar_song])
#=============================================================================

## 步驟 2：下載並整理歌單資料

```python
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')
lines = data.read().decode('utf-8').split('\n')[2:]
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
print(playlists[0])
```

### 這份資料是什麼

這是一份電台歌單資料集。`train.txt` 的**每一行是一份歌單**，裡面是用空格隔開的**歌曲編號**，例如（示意）：

```
0 1 2 3 4 5 6 7 8 9 ...
```

### 逐步拆解

| 步驟 | 說明 |
|---|---|
| `request.urlopen(網址)` | 開啟網路連線，取得檔案 |
| `data.read()` | 讀取全部內容（得到 bytes，也就是原始位元組） |
| `.decode('utf-8')` | 把 bytes 轉成文字 |
| `.split('\n')` | 依換行切成一行一行（得到 list） |
| `[2:]` | **跳過前兩行**（檔案開頭是說明文字，不是歌單） |

### 這個 list comprehension 怎麼讀

```python
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
```

展開來等價於：

```python
playlists = []
for s in lines:
    if len(s.split()) > 1:               # 這份歌單至少要有 2 首歌
        playlists.append(s.rstrip().split())
```

| 部分 | 說明 |
|---|---|
| `s.split()` | 依空白切成一首一首歌（得到編號的 list） |
| `if len(s.split()) > 1` | **過濾掉只有 0 或 1 首歌的行**。只有一首歌的歌單提供不了「誰和誰一起出現」的資訊 |
| `s.rstrip()` | 去掉行尾的空白或換行 |

### 結果的樣子

`playlists` 是「**list 的 list**」：外層是歌單，內層是一份歌單裡的歌曲編號（**字串**）：

```python
[['0', '1', '2', ...],     ← 第 1 份歌單
 ['100', '57', '3', ...],  ← 第 2 份歌單
 ...]
```

**編號是字串**，所以後面查詢時要 `str(song_id)`。

這其實就是 Word2Vec 要的輸入格式：「很多個句子，每個句子是一串詞（字串）」。歌單 = 句子，歌曲編號 = 詞。

`print(playlists[0])` 印出第一份歌單，讓你確認格式是對的。歌單數量可以用 `len(playlists)` 確認。

> 這段程式**每次執行都會重新下載**。想省時間，可以把資料存到本機（見實驗區）。

In [2]:
data = request.urlopen(
    'https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt'
)

lines = data.read().decode('utf-8').split('\n')[2:]

playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
print(playlists[0])

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43']


## 步驟 3：下載歌曲資訊（編號對照歌名）

歌單裡只有**編號**，人看不懂。`song_hash.txt` 是「編號 → 歌名、歌手」的對照表。

```python
songs_file = request.urlopen('.../song_hash.txt')
songs_file = songs_file.read().decode('utf-8').split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data = songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')
```

### 逐步拆解

| 步驟 | 說明 |
|---|---|
| `.split('\t')` | 這個檔案用 **Tab** 分隔欄位（不是空格），因為歌名裡本來就有空格 |
| `pd.DataFrame(data = songs, columns = [...])` | 把「list 的 list」變成表格，欄位叫 `id`、`title`、`artist` |
| `.set_index('id')` | **把 `id` 欄位變成列的標籤（index）**，之後可以用編號查詢 |

### `set_index` 的意義

| | 設定前 | 設定後 |
|---|---|---|
| 查「編號 2172」的歌 | 要用條件篩選 `df[df['id'] == '2172']` | 直接 `df.loc['2172']` |
| index | 0, 1, 2, ...（自動產生的位置） | `'0'`, `'1'`, `'2'`, ...（歌曲編號，字串） |

### 值得留意的小地方

- 檔案以換行結尾的話，`split('\n')` 最後可能多出一個空字串，變成 `id` 為空的一列。這通常無害，我沒有確認這份資料是否如此，可以用 `songs_df.tail()` 看看。
- 編號是**字串**（`'2172'`），不是整數 `2172`。

`print(songs_df)` 印出整張表，確認格式正確。曲目數量可以用 `len(songs_df)` 確認。

In [ ]:
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode('utf-8').split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data = songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

print(songs_df)

## 步驟 4：訓練 Word2Vec

```python
model = Word2Vec(
    playlists,
    vector_size=32,
    window=20,
    negative=50,
    min_count=1,
    workers=4
)
```

### 訓練在做什麼

Word2Vec 在歌單裡「滑動」一個視窗：

```
一份歌單：  s1  s2  s3  [s4]  s5  s6  s7
                        ↑ 目標歌曲
視窗內的歌 = 它的「上下文」
```

它學習讓「**目標歌曲的向量**」和「**上下文歌曲的向量**」彼此接近。同一份歌單裡常一起出現的歌，向量就越來越靠近。

### 參數說明

| 參數 | 值 | 說明 |
|---|---|---|
| `playlists` | 歌單清單 | 訓練資料，格式是「很多個句子，每句是一串字串」 |
| `vector_size=32` | 32 | 每首歌用 **32 個數字**表示（向量維度）。維度大能表達更多細節，但需要更多資料 |
| `window=20` | 20 | 上下文視窗大小：目標歌曲前後各看 **20 首**。一般文字常用 5 左右；歌單裡的順序不那麼嚴格，所以用大視窗，視為「同一份歌單裡的歌都有關聯」 |
| `negative=50` | 50 | **負採樣**：每學一個「正確配對」，就隨機抽 50 首**沒關係的歌**當反例，讓模型學會區分 |
| `min_count=1` | 1 | 只出現 1 次的歌也保留（預設會丟掉出現次數太少的詞） |
| `workers=4` | 4 | 用 4 個執行緒平行訓練 |

### 負採樣是什麼

如果每次都要在 7 萬多首歌裡算「正確答案是哪一首」，太慢。負採樣把問題改成簡單的二選一：

```
正確配對：  (目標歌, 同歌單的歌)   → 要預測「相關」
負樣本：    (目標歌, 隨機抽的歌)   → 要預測「無關」   × 50 組
```

### 沒有寫出來的預設值（很重要）

| 參數 | 預設值 | 意思 |
|---|---|---|
| `sg` | 0 | 我實測（gensim 4.4.0）預設是 **0 = CBOW**：用上下文預測中間的歌。設成 1 是 Skip-gram：用中間的歌預測上下文 |
| `epochs` | 5 | 整份資料**看 5 遍**（同樣是 gensim 4.4.0 實測，你的版本可能不同） |

### 結果

訓練完，`model.wv` 就存著每首歌的 32 維向量：

```python
model.wv['2172']          # 形狀 (32,)，這首歌的向量
len(model.wv)             # 有幾首歌（詞彙表大小）
```

### 可重現性

`workers=4` 使用多執行緒，加上隨機初始化，**每次訓練的結果會略有不同**，也就是每次推薦的 5 首歌可能不完全一樣。想讓結果更穩定，需要設定 `seed`，並把 `workers` 改成 1（用多執行緒時，即使有 `seed` 也不能保證每次相同，這是我依 gensim 文件的印象寫的）。

### 訓練要多久

歌單資料量很大的話，需要數十秒到數分鐘，視你的 CPU 而定。我沒有實測你的資料。

In [ ]:
model = Word2Vec(
    playlists,
    vector_size=32,
    window=20,
    negative=50,
    min_count=1,
    workers=4
)

## 步驟 5：查詢某一首歌最像的歌

```python
song_id = 2172
print(model.wv.most_similar(positive=str(song_id)))
print(songs_df.iloc[2172])
```

### 第一行：找出最像的歌

`most_similar` 沒有指定 `topn`，預設回傳最像的 **10 首**，格式是 list，每個元素是 `(歌曲編號, 相似度)`：

```python
[('3167', 0.83), ('2147', 0.81), ...]      # 相似度的數字只是示意
```

| 部分 | 說明 |
|---|---|
| 歌曲編號 | **字串**（和歌單裡的格式一樣） |
| 相似度 | 餘弦相似度，介於 -1 到 1，**越接近 1 越相似** |

**餘弦相似度**是看兩個向量「方向」有多接近，不管長度。直覺上就是兩個箭頭夾角有多小。

### 第二行：這首歌本身是什麼

`songs_df.iloc[2172]` 取出「第 2172 列」，也就是這首歌的資訊（歌名、歌手）。因為編號和位置一致（見步驟 1 的說明），所以 `iloc[2172]` 剛好就是編號 2172 的歌。更穩妥的寫法是 `songs_df.loc['2172']`。

### 用一個小實驗看懂 Word2Vec 在做什麼

以下是我用**合成資料**實測的結果（gensim 4.4.0）：兩種「風格」的歌，A 類（編號 1~4）只會和 A 類出現在同一份歌單，B 類（編號 11~14）只會和 B 類一起。訓練後：

```
most_similar('1'):       most_similar('11'):
   4   0.777                14   0.817
   3   0.719                12   0.764
   2   0.705                13   0.760
  11   0.144                 3   0.185     ← 另一類的歌，相似度很低
```

**同一類的歌彼此相似度高（約 0.7~0.8），不同類的歌相似度低（約 0.1~0.2）。** 完全沒有告訴模型「這是什麼風格」，它只靠「誰和誰一起出現」就學到了。你的真實資料也是同樣的道理，只是規模大得多。

### 一個可能的錯誤

如果查詢的編號不在詞彙表裡（例如這首歌從沒出現在任何有 2 首以上的歌單中），`most_similar` 會報 `KeyError`，訊息是找不到這個 key。我印象中 gensim 4 是這樣，沒有針對你的版本確認。

In [ ]:
song_id = 2172
print(model.wv.most_similar(positive=str(song_id)))
print(songs_df.iloc[2172])

## 步驟 6：用推薦函式輸出歌曲清單

```python
print(print_recommendation(2172))
```

呼叫步驟 1 定義的函式。它做的事：

```
song_id = 2172
   │ most_similar(topn=5)
   ▼
[('3167', 0.83), ('2147', 0.81), ...]        ← 最像的 5 首（編號, 相似度）
   │ np.array(...)[:, 0]
   ▼
['3167', '2147', ...]                          ← 只留編號
   │ songs_df.iloc[...]
   ▼
表格：這 5 首歌的 title 與 artist
```

輸出是一個有 `title` 和 `artist` 兩欄的表格，每一列是一首被推薦的歌，列標籤是歌曲編號。

### 怎麼判斷推薦得好不好

你可以用常識檢查：

| 現象 | 判斷 |
|---|---|
| 推薦的歌風格、歌手和原曲接近 | 推薦合理 |
| 推薦了毫不相干的歌 | 資料量不足、參數不合適，或歌單本身就很雜 |

這個方法的特色是：**它只知道「常被一起播放」，不知道歌的內容**。所以它會推薦「同一個電台常排在一起的歌」，不一定是曲風相同的歌。具體的推薦結果請以你跑出來的為準，我沒有實際跑這份資料。

In [ ]:
print(print_recommendation(2172))

---

# 重點整理

## 一、整份程式一句話

> 把電台歌單當成句子、歌曲編號當成單字，用 Word2Vec 學出每首歌的 32 維向量；給一首歌，就找向量最接近的 5 首歌，查出歌名與歌手，作為推薦。

## 二、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | embedding（嵌入） | 用一串數字（向量）表示一個事物，距離近 = 相似 |
| 2 | Word2Vec | 靠「誰和誰常一起出現」學習向量 |
| 3 | item2vec | 把 Word2Vec 用在歌、商品等非文字項目上 |
| 4 | 歌單 = 句子 | Word2Vec 的輸入是「很多串字串」 |
| 5 | `window` | 上下文視窗大小，決定「多近算一起出現」 |
| 6 | 負採樣（`negative`） | 每個正確配對，配上若干隨機反例，讓模型學會區分 |
| 7 | `vector_size` | 向量的維度，越大越細緻但需要更多資料 |
| 8 | 餘弦相似度 | 比較兩個向量的方向，越接近 1 越相似 |
| 9 | `model.wv` | 訓練完存放所有向量的地方 |
| 10 | `.loc` vs `.iloc` | `.loc` 用標籤、`.iloc` 用位置，不要混用 |

## 三、程式流程速記

```
1. 定義 print_recommendation（用到後面才建立的 model 與 songs_df）
2. 下載 train.txt → 切成一份份歌單（至少 2 首歌）→ playlists
3. 下載 song_hash.txt → 用 Tab 切 → DataFrame → set_index('id')
4. Word2Vec(playlists, vector_size=32, window=20, negative=50, ...)
5. model.wv.most_similar(positive='2172')   → (編號, 相似度)
6. print_recommendation(2172)               → 最像的 5 首歌的 title 與 artist
```

## 四、最容易搞混的地方

1. **為什麼歌曲編號要 `str(...)`？**
   歌單切出來的編號是字串，Word2Vec 的詞彙表裡也是字串，用整數去查會找不到。

2. **`[:, 0]` 是什麼意思？**
   二維陣列的「所有列、第 0 欄」。`most_similar` 的每個元素是 `(編號, 相似度)`，`[:, 0]` 只取編號。

3. **為什麼 `np.array` 之後相似度也變字串了？**
   陣列只能有一種資料型別，編號是字串，整個陣列就被轉成字串型別。這裡不需要相似度，所以沒問題。

4. **`window=20` 為什麼這麼大？**
   歌單裡的順序不像句子那麼嚴格，整份歌單的歌都算有關聯，所以用大視窗。

5. **`.iloc` 和 `.loc` 差在哪？為什麼你的程式用 `.iloc` 也對？**
   `.iloc` 看位置、`.loc` 看標籤。你的程式碰巧「位置 = 編號」，所以兩者結果相同，但這是巧合。

6. **這個模型知道歌的內容嗎？**
   不知道，它只知道「誰和誰常一起出現」。

7. **每次推薦的結果都一樣嗎？**
   不一定。多執行緒與隨機初始化會讓每次訓練略有不同。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `KeyError: ... not present in vocabulary` | 查詢的歌曲編號不在詞彙表裡 | 確認編號存在於 `model.wv` |
| 下載失敗、`URLError` | 沒有網路，或資料來源失效 | 檢查網路；我沒有確認該網址目前是否仍可用 |
| 推薦的歌明顯不相干 | 用了 `.iloc` 但位置與編號不一致 | 改用 `.loc`；檢查 `songs_df` 有沒有被重新排序或刪列 |
| `most_similar` 找不到字串編號 | 傳了整數 | 轉成字串 `str(song_id)` |
| 每次推薦結果不同 | 隨機性 | 設定 `seed` 並把 `workers` 設為 1 |
| 訓練很慢 | 資料量大 | 調低 `epochs`、`window`、`negative`，或增加 `workers` |
| `ModuleNotFoundError: gensim` | 還沒安裝 | `pip install gensim` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **`.iloc` 與字串編號**
   我實測它會把字串當成整數位置，結果正確是因為「位置 = 編號」。更穩妥的是 `.loc`。

2. **函式名稱與行為不符**
   `print_recommendation` 沒有 print，而是回傳 DataFrame，叫 `get_recommendation` 更貼切。

3. **函式依賴全域變數**
   `model` 與 `songs_df` 沒有放在參數裡，是隱藏的依賴。

4. **每次執行都重新下載資料**
   既慢又依賴網路，建議存到本機。

5. **`urlopen` 沒有關閉連線**
   更好的寫法是用 `with request.urlopen(...) as f:`。

6. **`songs_df` 可能有一列空的編號**
   `split('\n')` 在檔案以換行結尾時會多出空字串。

7. **`min_count=1`**
   只出現一次的歌也保留，它的向量只被訓練到一點點，通常不太可靠。

8. **沒有評估推薦的品質**
   只能靠肉眼判斷。可以留一部分歌單當測試集，檢查模型能否預測出歌單裡的下一首歌。

9. **結果不可完全重現**
   沒有設定 `seed`，`workers=4` 的多執行緒也會引入隨機性。

10. **Python 3.13 與 gensim 的相容性**
    我沒有確認，你已經能執行的話就沒問題。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看資料的規模
print("歌單數:", len(playlists))
print("歌曲數:", len(songs_df))
print("詞彙表大小:", len(model.wv))
print("歌單長度平均:", sum(len(p) for p in playlists) / len(playlists))

# 2. 用 .loc 取代 .iloc（更穩妥）
def get_recommendation(song_id, topn=5):
    keys = [k for k, _ in model.wv.most_similar(positive=str(song_id), topn=topn)]
    return songs_df.loc[keys]

# 3. 看兩首歌的相似度
print(model.wv.similarity('2172', '3167'))

# 4. 向量運算：多首歌一起當基準（找出「同時像 A 和 B」的歌）
model.wv.most_similar(positive=['2172', '3167'], topn=5)

# 5. 排除某首歌（找「像 A 但不像 B」的歌）
model.wv.most_similar(positive=['2172'], negative=['3167'], topn=5)

# 6. 比較 CBOW 與 Skip-gram
model_sg = Word2Vec(playlists, vector_size=32, window=20, negative=50,
                    min_count=1, workers=4, sg=1)

# 7. 調整向量維度（16、32、64、128），觀察推薦是否更合理

# 8. 增加訓練遍數
model_long = Word2Vec(playlists, vector_size=32, window=20, negative=50,
                      min_count=1, workers=4, epochs=20)

# 9. 把下載的資料存到本機，之後不用重新下載
with open("train.txt", "wb") as f:
    f.write(request.urlopen(
        'https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt').read())

# 10. 儲存與載入訓練好的模型
model.save("song_w2v.model")
loaded = Word2Vec.load("song_w2v.model")
```

## 八、Word2Vec 與你之前學過的 embedding 對照

| | **Word2Vec（這份）** | Sentence-Transformers（微調 embedding 那幾份） | LLM（QLoRA、DPO、Phi-3） |
|---|---|---|---|
| 向量對象 | 單一項目 | 一整句話 | 每個 token（上下文相關） |
| 是否隨上下文變化 | **否（固定向量）** | 是 | 是 |
| 訓練成本 | **極低** | 中 | 高 |
| 訓練資料 | 共現關係 | 句子對、標籤、配對 | 大量文字 |
| 典型用途 | **推薦系統** | 語意搜尋 | 生成、對話 |

共通的主軸：**用向量表示事物，距離近就代表相似**。

## 九、一句話記住

> **Word2Vec = 常一起出現的東西，向量就靠近。** 把歌單當句子、歌曲當單字，不用知道歌的內容，只靠「誰和誰常被一起播放」，就能學出每首歌的向量，並用最近鄰做推薦。